In [ ]:
# Imports and set up constant variable names

import importlib
import sys
import os
sys.path.append(os.path.abspath(".."))

import PythonCode.LoadTeam
import PythonCode.ConvertToShowdown

importlib.reload(PythonCode.LoadTeam)
importlib.reload(PythonCode.ConvertToShowdown)

import requests
import json
from UniversalFunctions import *
from PythonCode.LoadTeam import LoadandValidateTeam
from PythonCode.ConvertToShowdown import ConvertTeamToShowdownJSON

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "qwen3:14b"

In [ ]:
# Create the request variable to edit and schemas for the AI to use when creating teams
TEAM_REQUEST = "Create a competitive Pokémon team that takes advantage of rain weather conditions"

TEAM_MEMBER_SCHEMA = {
    "type": "array",
    "minItems": 6,
    "maxItems": 6,
    "items": {"type": "string"}
}
TEAM_SCHEMA = {
    "type": "array",
    "minItems": 6,
    "maxItems": 6,
    "items": {
        "type": "object",
        "required": [
            "name",
            "moves",
            "ability",
            "item",
            "nature",
            "gender",
            "evs",
            "ivs"
        ],
        "properties": {
            "name": {"type": "string"},
            "moves": {
                "type": "array",
                "minItems": 4,
                "maxItems": 4,
                "items": {"type": "string"}
            },
            "ability": {"type": "string"},
            "item": {"type": "string"},
            "nature": {"type": "string"},
            "gender": {"type": "string"},
            "evs": {
                "type": "object",
                "required": ["hp", "atk", "def", "spa", "spd", "spe"],
                "properties": {
                    "hp": {"type": "integer"},
                    "atk": {"type": "integer"},
                    "def": {"type": "integer"},
                    "spa": {"type": "integer"},
                    "spd": {"type": "integer"},
                    "spe": {"type": "integer"}
                }
            },
            "ivs": {
                "type": "object",
                "required": ["hp", "atk", "def", "spa", "spd", "spe"],
                "properties": {
                    "hp": {"type": "integer"},
                    "atk": {"type": "integer"},
                    "def": {"type": "integer"},
                    "spa": {"type": "integer"},
                    "spd": {"type": "integer"},
                    "spe": {"type": "integer"}
                }
            }
        }
    }
}

In [ ]:
# Function to create the schema json
def BuildTeamSchema(chosenPokemon, regulationPool, items):
    lookup = {p["name"]: p for p in regulationPool}

    itemNames = items
    if items and isinstance(items[0], dict):
        itemNames = [item["name"] for item in items]

    teamItems = []

    for pokemon in chosenPokemon:
        p = lookup[pokemon]

        legalAbilities = sorted(a["name"] for a in p.get("abilities", []))
        legalMoves = sorted(p.get("moveset", []))

        teamItems.append({
            "type": "object",
            "required": [
                "name",
                "moves",
                "ability",
                "item",
                "nature",
                "gender",
                "evs",
                "ivs"
            ],
            "properties": {
                "name": {
                    "type": "string",
                    "enum": [p["name"]]
                },
                "moves": {
                    "type": "array",
                    "minItems": 4,
                    "maxItems": 4,
                    "items": {
                        "type": "string",
                        "enum": legalMoves
                    }
                },
                "ability": {
                    "type": "string",
                    "enum": legalAbilities
                },
                "item": {
                    "type": "string",
                    "enum": sorted(itemNames)
                },
                "nature": {
                    "type": "string",
                    "enum": [
                        "hardy", "lonely", "brave", "adamant", "naughty",
                        "bold", "docile", "relaxed", "impish", "lax",
                        "timid", "hasty", "serious", "jolly", "naive",
                        "modest", "mild", "quiet", "bashful", "rash",
                        "calm", "gentle", "sassy", "careful", "quirky"
                    ]
                },
                "gender": {
                    "type": "string",
                    "enum": ["", "M", "F"]
                },
                "evs": {
                    "type": "object",
                    "required": ["hp", "atk", "def", "spa", "spd", "spe"],
                    "properties": {
                        "hp": {"type": "integer", "minimum": 0, "maximum": 252},
                        "atk": {"type": "integer", "minimum": 0, "maximum": 252},
                        "def": {"type": "integer", "minimum": 0, "maximum": 252},
                        "spa": {"type": "integer", "minimum": 0, "maximum": 252},
                        "spd": {"type": "integer", "minimum": 0, "maximum": 252},
                        "spe": {"type": "integer", "minimum": 0, "maximum": 252}
                    }
                },
                "ivs": {
                    "type": "object",
                    "required": ["hp", "atk", "def", "spa", "spd", "spe"],
                    "properties": {
                        "hp": {"type": "integer", "enum": [31]},
                        "atk": {"type": "integer", "enum": [31]},
                        "def": {"type": "integer", "enum": [31]},
                        "spa": {"type": "integer", "enum": [31]},
                        "spd": {"type": "integer", "enum": [31]},
                        "spe": {"type": "integer", "enum": [31]}
                    }
                }
            }
        })

    return {
        "type": "array",
        "minItems": 6,
        "maxItems": 6,
        "prefixItems": teamItems
    }

In [ ]:
# Function for the AI

# Function to call the model
def CreateOllamaModel(systemPrompt, userPrompt, schema):
    model = {
        "model": MODEL_NAME,
        "stream": False,
        "format": schema,
        "messages": [
            {"role": "system", "content": systemPrompt.strip()},
            {"role": "user", "content": userPrompt.strip()}
        ],
        "options": {
            "temperature": 0
        }
    }

    try:
        response = requests.post(OLLAMA_URL, json=model, timeout=900)
        response.raise_for_status()
        data = response.json()
        return json.loads(data["message"]["content"])
    except requests.exceptions.ReadTimeout:
        raise RuntimeError("Ollama took too long to respond.")
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"Ollama request failed: {e}")
    except json.JSONDecodeError as e:
        raise RuntimeError(f"Ollama returned invalid JSON: {e}")

# Function to give the prompt to the AI to get the pokemon
def GetLegalPokemonForOllama(regulationPool, rules, teamRequest=""):
    legalPokemon = sorted(p["name"] for p in regulationPool)
    restrictedPokemon = sorted(p["name"] for p in regulationPool if p.get("restricted_regulation_i", False))
    
    systemPrompt = f"""
    You are building a Pokemon team for {rules['name']}.
    
    Return ONLY a JSON array of exactly 6 pokemon names.
    
    Rules:
    - Use only names from the provided legal species list.
    - No duplicate pokemon.
    - Maximum {rules['max_restricted']} restricted pokemon.
    - Use lowercase names with hyphens exactly as provided.
    - Do not include explanations.
    - Do not include markdown.
    """
    
    userPrompt = f"""
    Legal Pokemon:
    {json.dumps(legalPokemon)}
    
    Restricted Species:
    {json.dumps(restrictedPokemon)}
    
    Team request:
    {teamRequest if teamRequest else "Choose one realistic competitive team of 6 for the format."}
    """
    
    return systemPrompt, userPrompt

# Create the pokemon team with the schema and create the prompt
def SetupPokemonSet(chosenPokemon, regulationPool, items, rules, teamRequest=""):
    lookup = {p["name"]: p for p in regulationPool}
    selectedData = []

    itemNames = items
    if items and isinstance(items[0], dict):
        itemNames = [item["name"] for item in items]

    for pokemon in chosenPokemon:
        p = lookup[pokemon]
        selectedData.append({
            "name": p["name"],
            "abilities": [a["name"] for a in p.get("abilities", [])],
            "moveset": p.get("moveset", [])
        })

    schema = BuildTeamSchema(chosenPokemon, regulationPool, itemNames)

    systemPrompt = f"""
You are filling in competitive sets for a Pokemon team in {rules['name']}.

Return ONLY a JSON array of exactly 6 Pokemon objects.

Rules:
- Use exactly the 6 provided Pokemon names.
- Do not change Pokemon names.
- Use only legal abilities for each Pokemon.
- Use only legal moves for each Pokemon.
- Use only items from the provided item list.
- No duplicate items.
- Exactly 4 moves per Pokemon.
- All IVs must be 31.
- Every EV must be an integer from 0 to 252.
- The total EVs for each Pokemon must be exactly 508 or 510.
- Prefer common legal spreads like 252/252/4.
- Nature must be lowercase.
- Gender must be "", "M", or "F".
- Use lowercase names, items, abilities, natures, and moves with hyphens.
- Do not include explanations.
- Do not include markdown.
"""

    userPrompt = f"""
Team request:
{teamRequest if teamRequest else "Use strong realistic competitive sets."}

Pokemon data:
{json.dumps(selectedData, separators=(",", ":"))}

Allowed items:
{json.dumps(sorted(itemNames), separators=(",", ":"))}
"""

    return systemPrompt, userPrompt, schema

# Function to fix and invalid teams the AI creates by running the team back through the AI
def FixIncorrectPokemonSets(currentTeam, chosenPokemon, regulationPool, items, rules, validationErrors, teamRequest=""):
    itemNames = items
    if items and isinstance(items[0], dict):
        itemNames = [item["name"] for item in items]

    schema = BuildTeamSchema(chosenPokemon, regulationPool, itemNames)

    systemPrompt = f"""
You are correcting a competitive Pokemon team for {rules["name"]}.

Return ONLY a JSON array of exactly 6 Pokemon objects.

Rules:
- Keep the same 6 Pokemon names.
- Fix all illegal moves, illegal abilities, illegal items, duplicate items, and illegal EV spreads.
- Use only legal abilities for each Pokemon.
- Use only legal moves for each Pokemon.
- Use only items from the provided item list.
- No duplicate items.
- Exactly 4 moves per Pokemon.
- All IVs must be 31.
- Every EV must be an integer from 0 to 252.
- The total EVs for each Pokemon must be exactly 508 or 510.
- Nature must be lowercase.
- Gender must be "", "M", or "F".
- Use lowercase names, items, abilities, natures, and moves with hyphens.
- Respect the team request as much as possible while keeping the team legal.
- Do not include explanations.
- Do not include markdown.
"""

    userPrompt = f"""
Team request:
{teamRequest if teamRequest else "Keep the team competitively reasonable."}

Current invalid team:
{json.dumps(currentTeam, separators=(",", ":"))}

Allowed items:
{json.dumps(sorted(itemNames), separators=(",", ":"))}

Validation errors:
{json.dumps(validationErrors, separators=(",", ":"))}
"""

    return CreateOllamaModel(systemPrompt, userPrompt, schema)

In [ ]:
# Functions to call the AI

def OllamaGenerateTeam(regulationPool, rules, teamRequest=""):
    systemPrompt, userPrompt = GetLegalPokemonForOllama(regulationPool, rules, teamRequest)
    return CreateOllamaModel(systemPrompt, userPrompt, TEAM_MEMBER_SCHEMA)

def GenerateSetsForPokemon(chosenPokemon, regulationPool, items, rules, teamRequest=""):
    systemPrompt, userPrompt, schema = SetupPokemonSet(chosenPokemon, regulationPool, items, rules, teamRequest)
    return CreateOllamaModel(systemPrompt, userPrompt, schema)

In [ ]:
# Files to create for the AI generated team
ollamaTeamFile = "../../data/GeneratedTeams/ollama_generated_team.json"
ollamaShowdownFile = "../../data/ShowdownFormattedTeams/ollama_generated_team_showdown.json"

# Get all the necessary data
regulationPool = LoadJSON("../../data/pokemon_regulation_i.json")
regulationRules = LoadJSON("../../data/regulation_i_rules.json")
items = LoadJSON("../../data/items.json")

# Get the AI generated team
chosenPokemon = OllamaGenerateTeam(regulationPool, regulationRules, TEAM_REQUEST)
generatedTeam = GenerateSetsForPokemon(chosenPokemon, regulationPool, items, regulationRules, TEAM_REQUEST)

# Write the team out to the JSON file and check if the team is valid
WriteJSON(ollamaTeamFile, generatedTeam)
validOllama = LoadandValidateTeam(ollamaTeamFile, regulationPool, regulationRules, items)

# If the team is not valid, rerun it through the AI to get a valid team. works within 3 tries max
attempt = 1
while not validOllama["valid"] and attempt <= 3:
    print(f"Attempt {attempt} errors:", validOllama["errors"])

    try:
        generatedTeam = FixIncorrectPokemonSets(
        generatedTeam,
        chosenPokemon,
        regulationPool,
        items,
        regulationRules,
        validOllama["errors"],
        TEAM_REQUEST
        )
    except RuntimeError as e:
        print(e)
        break

    WriteJSON(ollamaTeamFile, generatedTeam)
    validOllama = LoadandValidateTeam(ollamaTeamFile, regulationPool, regulationRules, items)
    attempt += 1

print(validOllama["errors"])

if validOllama["valid"]:
    showdownTeam = ConvertTeamToShowdownJSON(validOllama["team"], regulationRules["battle_level"])
    WriteJSON(ollamaShowdownFile, showdownTeam)
    print(f"Generated team saved to {ollamaTeamFile}")
    print(f"Showdown team saved to {ollamaShowdownFile}")
else:
    print("Team is still invalid after retries.")

Attempt 1 errors: ['Duplicate item not allowed: choice-specs']
[]
Generated team saved to ../../data/GeneratedTeams/ollama_generated_team.json
Showdown team saved to ../../data/ShowdownFormattedTeams/ollama_generated_team_showdown.json
